# 02-Embeddings — Geração de representações vetoriais

## Visão geral

Este módulo transforma cada documento em um **vetor denso de dimensão fixa**
que captura seu significado semântico. Esses vetores alimentam:

- **01-preprocessing** — detecção de near-duplicates via cosine similarity
- **04-topic-modeling** (BERTopic) — clustering semântico dos posts
- **03-sentiment** — features para classificação de sentimento

## Modelo padrão: Qwen3-Embedding 0.6B

| Parâmetro | Valor |
|---|---|
| Modelo | `qwen3-embedding:0.6b` (Ollama local) |
| Dimensão | **1024d nativo** (`dimension: null` no params.yaml) |
| Janela | 32.000 tokens |
| Matryoshka | Desabilitado — `dimension=None` usa o vetor completo |
| Backend | Ollama HTTP (`localhost:11434`) |

> `dimension: null` no params.yaml significa "usar dimensão nativa do modelo".
> Para Qwen3 0.6B, isso é **1024 dimensões**. A truncação Matryoshka
> (que reduziria para 768d) está desabilitada.


In [1]:
import asyncio
import httpx
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()

with open("../configs/params.yaml", encoding="utf-8") as f:
    params = yaml.safe_load(f)

CORPUS     = params.get("default_corpus", "arj")
corpus_cfg = params["corpora"][CORPUS]
TEXT_COL   = corpus_cfg["text_column"]          # chave real do params.yaml
SUBDIR     = corpus_cfg.get("subdir", CORPUS)
SEED       = params.get("seed", 42)

model_cfg  = params["embeddings"]["models"][0]   # modelo ativo
MODEL_NAME = model_cfg["name"]                   # "qwen3-embedding:0.6b"
DIMENSION  = model_cfg.get("dimension", None)    # None = nativo 1024d
SUFFIX     = model_cfg.get("suffix", "1024d")
TIMEOUT    = model_cfg.get("timeout", 120)

from datetime import datetime as _dt
INPUT_DIR  = Path("../data/input") / SUBDIR          # fallback legado

# Le o corpus direto do output do 01-preprocessing (latest-wins, sem copia)
_PRE_OUT = Path("../../01-preprocessing/data/output") / SUBDIR
def _resolve_latest(base, contains, fallback):
    base = Path(base)
    cands = [d for d in base.iterdir() if d.is_dir() and (d / contains).exists()] if base.exists() else []
    if cands:
        return max(cands, key=lambda d: d.name)
    if base.exists() and (base / contains).exists():
        return base
    return Path(fallback)
CORPUS_DIR = _resolve_latest(_PRE_OUT, "corpus_limpo.csv", INPUT_DIR)
print(f"Corpus de : {CORPUS_DIR}")

# Saida versionada: subpasta carimbada <subdir>_<AAAAMMDD_HHMMSS>
OUTPUT_BASE = Path("../data/output") / SUBDIR
OUTPUT_DIR  = OUTPUT_BASE / f"{SUBDIR}_{_dt.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAFE_NAME  = MODEL_NAME.replace("/", "_").replace(":", "_")
CACHE_PATH = OUTPUT_DIR / f"embeddings_{SAFE_NAME}_{SUFFIX}.npy"

print(f"Corpus    : {CORPUS}")
print(f"Modelo    : {MODEL_NAME}")
print(f"Dimensão  : {DIMENSION or 'nativa (1024d)'}")
print(f"Cache     : {CACHE_PATH.name}")


Corpus de : ..\..\01-preprocessing\data\output\tweets_bre2022\tweets_bre2022_20260629_215159
Corpus    : tweets_bre2022
Modelo    : qwen3-embedding:8b
Dimensão  : nativa (1024d)
Cache     : embeddings_qwen3-embedding_8b_4096d.npy


## Carregar corpus

Lê `data/input/corpus_limpo.csv` (gerado pelo 01-preprocessing). Coluna do
texto: `message` (schema unificado pós-rename).


In [2]:
corpus_path = CORPUS_DIR / "corpus_limpo.csv"
df = pd.read_csv(corpus_path, encoding="utf-8")
docs = df[TEXT_COL].astype(str).tolist()

print(f"Corpus: {len(docs)} documentos")
print(f"Coluna de texto: {TEXT_COL!r}")
print(f"Primeiro doc (120 chars): {docs[0][:120]!r}")
print()
print("Estatísticas de comprimento (chars):")
lens = pd.Series([len(d) for d in docs])
print(lens.describe().round(1).to_string())

Corpus: 8811 documentos
Coluna de texto: 'message'
Primeiro doc (120 chars): '@LulaOficial “Lulinha”, “Alckmin”, “Paulo Freire”, “Faz o L”, “#LADRAONOJN” “É ELE” e “BOBO DA CORTE” foram alguns dos t'

Estatísticas de comprimento (chars):
count    8811.0
mean      163.7
std        85.3
min        21.0
25%        96.0
50%       147.0
75%       228.5
max      1287.0


## Verificar disponibilidade do Ollama

Se algum modelo no `params.yaml` é do tipo `"ollama"`, precisamos do servidor
Ollama rodando localmente em `http://localhost:11434`. A célula abaixo faz um
ping rápido e lista os modelos disponíveis.

Para iniciar o Ollama: `ollama serve` em outro terminal.
Para baixar o modelo padrão: `ollama pull qwen3-embedding:0.6b`.


In [3]:
import httpx

ollama_models = [m for m in params["embeddings"]["models"] if m.get("type") == "ollama"]
print(f"Modelos Ollama configurados: {len(ollama_models)}")

if ollama_models:
    try:
        r = httpx.get("http://localhost:11434/api/tags", timeout=3.0)
        r.raise_for_status()
        installed = [m["name"] for m in r.json().get("models", [])]
        print(f"Ollama OK — {len(installed)} modelos instalados:")
        for m in installed[:10]:
            print(f"  - {m}")

        # Verificar que cada modelo configurado está disponível
        print()
        for cfg in ollama_models:
            name = cfg["name"]
            mark = "✓" if name in installed else "✗ AUSENTE"
            print(f"  {mark} {name}")
    except Exception as e:
        print(f"Ollama indisponível: {type(e).__name__}: {e}")
        print("  Para usar modelos Ollama, execute em outro terminal:")
        print("    ollama serve")


Modelos Ollama configurados: 1
Ollama OK — 4 modelos instalados:
  - qwen3-embedding:8b
  - gemma2:2b-instruct-q4_K_M
  - qwen3-embedding:4b
  - qwen3-embedding:0.6b

  ✓ qwen3-embedding:8b


## Geração de embeddings

Para cada modelo configurado:
1. Verifica cache em `data/output/embeddings_<modelo>_<sufixo>.npy`
2. Se ausente ou desalinhado, computa via backend apropriado:
   - `sentence_transformers` → `SentenceTransformer.encode()` (local)
   - `ollama` → HTTP async com concorrência limitada (5 paralelos)
3. **Truncação Matryoshka** quando `dimension < dimensão nativa` (corta
   primeiras D componentes + renormaliza L2).
4. Salva em disco (cache reutilizável).

Modelos Ollama indisponíveis são pulados silenciosamente (não bloqueiam o resto).


In [ ]:
async def _get_embedding(client, text, model, dimension, timeout):
    resp = await client.post(
        "http://localhost:11434/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=timeout
    )
    emb = resp.json()["embedding"]
    if dimension is not None and len(emb) > dimension:
        emb = emb[:dimension]
    arr = np.array(emb, dtype=np.float32)
    return arr / np.linalg.norm(arr)

async def _batch_embeddings(texts, model, dimension, max_concurrent=5, timeout=120):
    sem = asyncio.Semaphore(max_concurrent)
    async with httpx.AsyncClient() as client:
        async def _one(text):
            async with sem:
                return await _get_embedding(client, text, model, dimension, timeout)
        return await asyncio.gather(*[_one(t) for t in texts])

def get_or_compute_embeddings(texts, model, cache_path, dimension=None, timeout=120):
    cache_path = Path(cache_path)
    expected_dim = dimension if dimension is not None else 1024  # Qwen3 nativo
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape[0] == len(texts) and arr.shape[1] == expected_dim:
            print(f"Cache hit : {cache_path.name}  shape={arr.shape}")
            return arr
        if arr.shape[0] == len(texts) and arr.shape[1] != expected_dim:
            print(f"Cache com dimensão errada ({arr.shape[1]}d ≠ {expected_dim}d), recomputando...")
        else:
            print(f"Cache desalinhado ({arr.shape[0]} != {len(texts)}), recomputando...")
    print(f"Computando {len(texts)} embeddings com '{model}'...")
    loop = asyncio.get_event_loop()
    vecs = loop.run_until_complete(_batch_embeddings(texts, model, dimension, timeout=timeout))
    arr  = np.stack(vecs).astype(np.float32)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, arr)
    print(f"Salvo     : {cache_path.name}  shape={arr.shape}")
    return arr

# --- Executar ---
corpus_path = CORPUS_DIR / "corpus_limpo.csv"
df   = pd.read_csv(corpus_path, encoding="utf-8")
docs = df[TEXT_COL].astype(str).tolist()
print(f"Documentos: {len(docs)}")

embeddings = get_or_compute_embeddings(docs, MODEL_NAME, CACHE_PATH, DIMENSION, TIMEOUT)


Documentos: 8811
Computando 8811 embeddings com 'qwen3-embedding:8b'...


In [ ]:
# Verificação de dimensão — deve ser 1024 para Qwen3 0.6B nativo
assert embeddings.shape[0] == len(docs), \
    f"Shape mismatch: {embeddings.shape[0]} embeddings para {len(docs)} docs"
assert embeddings.shape[1] == 4096, \
    f"Dimensão inesperada: {embeddings.shape[1]}d (esperado 4096d nativo Qwen3)"
print(f"OK: {embeddings.shape}  — 1024d nativo confirmado")
print(f"Tamanho em disco: {embeddings.nbytes / 1e6:.1f} MB")


## Benchmark — propriedades do modelo

Tabela com nome, shape, tamanho em disco e norma L2 média dos embeddings gerados.
Norma média ≈ 1.0 confirma normalização L2 correta (esperado para Qwen3 via Ollama
com renormalização pós-truncagem — no nosso caso, sem truncagem, apenas normalização).

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)
benchmark = pd.DataFrame([{
    "modelo":      MODEL_NAME,
    "shape":       f"{embeddings.shape[0]}×{embeddings.shape[1]}",
    "tamanho_MB":  embeddings.nbytes / (1024 ** 2),
    "norma_média": float(norms.mean()),
    "norma_std":   float(norms.std()),
}])
print(benchmark.round(4).to_string(index=False))

## Visualização — UMAP 2D dos embeddings

Reduz os embeddings para 2 dimensões via UMAP e plota os documentos. Permite
inspecionar visualmente:

- **Aglomerados** — regiões densas indicam temas potencialmente coesos
- **Outliers isolados** — pontos longe de tudo (candidatos a noise no BERTopic)
- **Pontes / continuum** — zonas onde temas se sobrepõem semanticamente

Salva o scatter em `data/output/embeddings_umap.png`.

In [ ]:
import matplotlib.pyplot as plt

try:
    from umap import UMAP
except ImportError:
    print("umap-learn não instalado. Para habilitar: pip install umap-learn")
else:
    n_neighbors = min(15, max(2, embeddings.shape[0] - 1))
    reducer = UMAP(
        n_components=2, n_neighbors=n_neighbors,
        min_dist=0.1, metric="cosine", random_state=SEED,
    )
    xy = reducer.fit_transform(embeddings)

    plt.figure(figsize=(8, 6))
    plt.scatter(xy[:, 0], xy[:, 1], s=10, alpha=0.6, c="steelblue", edgecolors="none")
    plt.title(f"{MODEL_NAME}  —  {embeddings.shape[0]} docs × {embeddings.shape[1]}d")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    out_png = OUTPUT_DIR / "embeddings_umap.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figura salva em {out_png}")

## Validação de qualidade — sanity checks

Verifica propriedades esperadas dos embeddings:

1. **Sem NaN ou Inf** — qualquer um indica problema na geração.
2. **Norma L2** — Sentence-Transformers normalizam por default; o Qwen3 via
   Ollama é renormalizado pela truncação Matryoshka. Esperamos média ≈ 1.0.
3. **Variância por dimensão** — se alguma dimensão é constante, o modelo está
   subutilizando o espaço.
4. **Distribuição da norma** — histograma para visualizar se há outliers.


In [ ]:
n_nan = int(np.isnan(embeddings).sum())
n_inf = int(np.isinf(embeddings).sum())
norms = np.linalg.norm(embeddings, axis=1)
var_per_dim = embeddings.var(axis=0)
n_const = int((var_per_dim < 1e-10).sum())

print(f"{'modelo':<45} {'NaN':>5} {'Inf':>5} {'norm_méd':>10} {'norm_std':>10} {'dim_const':>10}")
print("-" * 95)
print(f"{MODEL_NAME:<45} {n_nan:>5} {n_inf:>5} {norms.mean():>10.4f} {norms.std():>10.4f} {n_const:>10}")

plt.figure(figsize=(8, 3))
plt.hist(norms, bins=30, color="steelblue", alpha=0.7, edgecolor="black")
plt.xlabel("Norma L2 do embedding")
plt.ylabel("Frequência")
plt.title(f"Distribuição da norma — {MODEL_NAME}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Próximos passos

Os embeddings em `data/output/` devem ser **copiados manualmente** para os
módulos downstream:

```bash
# Para o BERTopic
cp ../data/output/embeddings_*.npy ../../04-topic-modeling/data/input/

# Para o 03-sentiment (se for usar)
cp ../data/output/embeddings_*.npy ../../03-sentiment/data/input/

# Para o 01-preprocessing (near-duplicate removal — re-rodar depois)
cp ../data/output/embeddings_*.npy ../../01-preprocessing/data/input/
```

Em seguida, abra `04-topic-modeling/notebooks/01_bertopic.ipynb` e execute
célula a célula. O notebook BERTopic detecta automaticamente o backend pelo
`params.yaml` (`embedding_backend: ollama` ou `sentence_transformers`).


In [ ]:
print("=" * 60)
print("RESUMO — 02-EMBEDDINGS")
print("=" * 60)
print(f"Documentos processados : {len(docs)}")
print(f"Modelo                 : {MODEL_NAME}")
print(f"Dimensão               : {embeddings.shape[1]}d")
print(f"Cache                  : {CACHE_PATH.name}")
print(f"Tamanho                : {embeddings.nbytes / 1e6:.1f} MB")
print()
print(f"Outputs em: {OUTPUT_DIR.resolve()}")